In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
from pathlib import Path
import pickle
import operator

import IPython.display as ipydisplay
import numpy as np
import scipy as sp
from openmm.app import PDBFile
from openbabel import pybel
import pandas as pd
from pdbfixer import PDBFixer

import sciapi
import scids
import scifile
import scishow
import caddpy

Define paths for storing intermediate results:

In [ ]:
project_id = "3w32"

cache_dir = Path(f".tmp/{project_id}")
assert " " not in str(cache_dir), "autogrid does not accept whitespace in paths"
pdb_filepath_raw = cache_dir / "receptor_raw.pdb"
pdb_filepath_fixed = cache_dir / "receptor_fixed.pdb"
pdb_filepath_apo = cache_dir / "receptor_fixed_apo.pdb"
pdbqt_filepath = cache_dir / "receptor.pdbqt"
pockets_filepath = cache_dir / "pockets.pkl"
autogrid_dirpath = cache_dir/ "autogrid"
autogrid_common_path = autogrid_dirpath / project_id
gpf_filepath = autogrid_common_path.with_suffix(".gpf")
cache_dir.mkdir(exist_ok=True, parents=True)
autogrid_dirpath.mkdir(exist_ok=True, parents=True)

## Structure Preparation

Obtain a PDB file:

In [ ]:
pdb_id = project_id

if not pdb_filepath_raw.is_file():
    pdb_file_content = sciapi.pdb.file.entry(pdb_id=pdb_id, file_format="pdb")
    pdb_filepath_raw.write_bytes(pdb_file_content)

Fix the PDB file:

In [ ]:
if not pdb_filepath_fixed.is_file():
    fixer = PDBFixer(filename=str(pdb_filepath_raw))
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(pdb_filepath_fixed, 'w'))

We store one fixed PDB with ligands, and one without.
The one with ligands is used later as reference for visualization and validation.
The one without ligands is used to create a PDBQT file for AutoGrid:

In [ ]:
if not pdb_filepath_apo.is_file():
    fixer = PDBFixer(filename=str(pdb_filepath_fixed))
    fixer.removeHeterogens(False)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(pdb_filepath_apo, 'w'))

Create the PDBQT file from the cleaned and fixed PDB file:

In [ ]:
if not pdbqt_filepath.is_file():
    molecule = next(pybel.readfile("pdb", str(pdb_filepath_apo)))
    molecule.calccharges("gasteiger")
    molecule.write(
        format="pdbqt",
        filename=str(pdbqt_filepath),
        overwrite=True,
        opt={"r": None, "n": None, "p": None, "h": None},
    )


Read the created PDBQT file (used later for atom type references):

In [ ]:
pdbqt_file = scifile.autodock_pdbqt.read(pdbqt_filepath)

## Binding Site Detection

Use DoGSiteScorer to detect binding sites (this is the longest step, taking ca. 1-2 min):

In [ ]:
if not pockets_filepath.is_file():
    protplus_upload_results = sciapi.proteinsplus().upload_pdb(pdb_filepath_fixed.read_bytes())
    dogsite_results = sciapi.proteinsplus().dogsite(
        pdb_id=protplus_upload_results.dummy_pdb_id,
        algorithm="scorer",
    )
    binding_sites = dogsite_results.full_data
    with open(pockets_filepath, "wb") as f:
        pickle.dump(binding_sites, f)
else:
    with open(pockets_filepath, "rb") as f:
        binding_sites = pickle.load(f)

Select the best binding pocket.
Here we select the binding pocket with a volume less than 1000 Å^3 and highest score:

In [ ]:
binding_site_df = pd.DataFrame(binding_sites)
idx_selected_pocket = binding_site_df[binding_site_df["volume"] < 1000]["simpleScore"].idxmax()
binding_site_df.loc[idx_selected_pocket, "name"]

Read the volume map for the selected pocket:

In [ ]:
pocket = scifile.mrc.read(binding_site_df.loc[idx_selected_pocket, "mrc"])
pocket

Create a `Grid` from the map data:

In [ ]:
grid = scids.grid.from_shape_spacing_anchor(
    shape=pocket.n_xyz,
    spacing=pocket.grid_vectors[0][0],
    anchor_type="lower",
    anchor_coord=pocket.grid_origin,
)
grid

## Energy Calculation

In [ ]:
ligand_types = ("C", "HD", "OA")
field_names = ligand_types + ("e", "d")
mif = caddpy.mif.autogrid.from_pdbqt(
    files=pdbqt_filepath,
    grid=grid,
    ligand_types=ligand_types,
    output_dir=autogrid_dirpath,
)

Plot energy distributions:

In [ ]:
field_colors = {
    "c":  [255, 165, 0],     # Hydrophobic (Orange)
    "a":  [128, 0, 128],     # Aromatic (Purple)
    "hd": [0, 255, 255],     # H-bond Donor (Cyan)
    "n":  [0, 0, 255],       # Non-Hbonding Nitrogen (Blue)
    "na": [30, 144, 255],    # H-bonding Nitrogen (Dodger Blue)
    "oa": [0, 128, 0],       # H-bonding Oxygen (Green)
    "sa": [255, 20, 147],    # H-bonding Sulphur (Deep Pink)
    "e":  [255, 0, 0],       # Electrostatic (Red)
    "d":  [255, 215, 0],     # Desolvation (Gold)
    "pi": [255, 0, 0],
    "ni": [0, 0, 255],
}

In [ ]:
scishow.matplotlib.Plotter(
).add_histograms(
    data=mif.tensor[:-1],
    axes=0,
    names=field_names[:-1],
    colors=[field_colors[field_name.lower()] for field_name in field_names[:-1]]
).show()

In [ ]:
tensor = np.array(mif.tensor)
tensor[:, pocket.data == 0] = np.nan

In [ ]:
scishow.matplotlib.Plotter(
).add_histograms(
    data=tensor[:-1],
    axes=0,
    names=field_names[:-1],
    colors=[field_colors[field_name.lower()] for field_name in field_names[:-1]],
    bin_range=(None, 5)
).show()

## Feature Selection

Use the pocket volume as a mask and set everything outside of it equal zero:

In [ ]:
mask_site = pocket.data == 1

Define selection criteria:

In [ ]:
field_names = ligand_types + ("PI", "NI")
field_indices = (0, 1, 2, 3, 3)
field_cutoffs = (-0.4, -0.35, -0.6, -1, 1)
cutoff_operators = (operator.lt, operator.lt, operator.lt, operator.lt, operator.gt)
structure = np.ones((3, 3, 3), dtype=int)  # full 26-connectivity

Generate features:

In [ ]:
features = []
for field_name, field_idx, field_cutoff, cutoff_operator in zip(field_names, field_indices, field_cutoffs, cutoff_operators):
    mask_field = cutoff_operator(mif.tensor[field_idx], field_cutoff)
    mask_final = np.logical_and(mask_field, mask_site)
    labeled_array, num_features = sp.ndimage.label(mask_final, structure=structure)
    if num_features == 0:
        continue
    sizes = np.bincount(labeled_array.ravel())[1:]
    centroids = np.array(sp.ndimage.center_of_mass(mask_final, labeled_array, range(1, num_features + 1)))
    centroids_coords = grid.index_coordinates(centroids)
    for feature_idx in range(num_features):
        feature_label = feature_idx + 1
        grid_selected = labeled_array == feature_label
        features.append(
            {
                "type": field_name,
                "size": sizes[feature_idx],
                "selected": sizes[feature_idx] >= 15,
                "center": centroids_coords[feature_idx],
                "coords": grid.coordinates[grid_selected],
            }
        )
feat_df = pd.DataFrame(features)
feat_df

## Visualization

All fields and features are added (hidden by default) under their respective atom type names. You can find them in the GUI. Toggle visibility for each field and vary the iso-level to analyze the field visually.

In [ ]:
nw = scishow.nglview.NGLWidget()
nw.add_component(str(pdb_filepath_fixed))
# Add the pocket
nw.add_volume(
    pocket.data,
    basis=pocket.grid_vectors,
    origin=pocket.grid_origin,
    name="pocket",
    representation_params=scishow.nglview.SurfaceRepresentationParameters(lazy=True, opacity=1, contour=True)
)
# Add the fields
for field_idx, field_name in enumerate(ligand_types + ("e", "d")):
    field_ = tensor[field_idx]
    nw.add_volume(
        field_,
        basis=mif.grid.unit_vectors,
        origin=mif.grid.lower_bounds,
        name=field_name,
        representation_params=scishow.nglview.SurfaceRepresentationParameters(
            lazy=True, opacity=1, visible=False, contour=True, isolevel=0, isolevel_type="value",
            isolevel_scroll=True, use_worker=True, color_value=f"rgb({", ".join(map(str,field_colors[field_name.lower()]))})"
        )
    )


voxel_volume = np.prod(grid.spacings)

for feat_idx, feature in enumerate(features):
    nw.add_spheres(
        coords=feature["coords"],
        colors=field_colors[feature["type"].lower()],
        radii=grid.spacings[0]/2,
        name=f"{feature["type"]}_{feat_idx}",
        representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=False)
    )
    if not feature["selected"]:
        continue
    center = feature["center"]
    size = feature["size"]
    total_volume = size * voxel_volume
    radii = (3 * total_volume / (4 * np.pi)) ** (1 / 3)
    nw.add_spheres(
        coords=center,
        colors=field_colors[feature["type"].lower()],
        radii=radii,
        name=f"{feature["type"]}_{feat_idx}_center",
        representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=True)
    )

# nw.add_representation_within_radius_of_selection(representation_type="spacefill")
nw.display(gui=True)

In [ ]:
poseview_interaction = sciapi.proteinsplus().poseedit("3w32", ["w32", "A", 1101])
ipydisplay.display(ipydisplay.SVG(poseview_interaction.svg))